In [1]:
import pickle
import numpy as np
import pandas as pd
from random import choice
from tqdm.auto import tqdm
from scipy.ndimage import label
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

In [2]:
L = 50
nsamples_per_p = 1_000
pp = np.arange(0.4, 0.8, 0.01)
alpha_values = [8., 6., 4., 2.]
MODEL_PATH = 'random_forest_percolation_model.joblib'

In [3]:
def find_neighbours(x, y, l):
    nbrs = []
    for dx, dy in [(0, -1), (-1, 0), (1, 0), (0, 1)]:
        x1, y1 = x + dx, y + dy
        if 0 <= x1 < l and 0 <= y1 < l:
            nbrs.append((x1, y1))
    return nbrs

def dfs(m, l):
    routes = [(n, 0) for n in range(l) if m[n, 0] == 1]
    end = [(n, l - 1) for n in range(l) if m[n, l - 1] == 1]
    m1 = m.copy()

    while routes:
        x, y = routes.pop()
        m1[x, y] = -1

        if (x, y) in end:
            return 1

        for xn, yn in find_neighbours(x, y, l):
            if m1[xn, yn] == 1 and (xn, yn) not in routes:
                routes.append((xn, yn))

    return 0

In [4]:
def random_maps():
    print("Генерация некоррелированных карт:")

    n_p_values = len(pp)
    mm_random = np.zeros((n_p_values, nsamples_per_p, L, L), dtype=np.float32)
    labels_random = np.zeros((n_p_values, nsamples_per_p), dtype=np.uint8)

    for i, p in enumerate(tqdm(pp, desc="p значения")):
        for j in range(nsamples_per_p):
            m = np.random.choice([0, 1], size=(L, L), p=[1-p, p])
            label_val = dfs(m, L)

            mm_random[i, j, :, :] = m
            labels_random[i, j] = label_val

    np.save("maps_random.npy", mm_random)
    np.save("labels_random.npy", labels_random)

    print("Сохранено: maps_random.npy")
    print("Сохранено: labels_random.npy")

    return mm_random, labels_random

In [11]:
def correlated_maps():
    print("Генерация коррелированных карт:")

    n_p_values = len(pp)
    n_alpha = len(alpha_values)

    mm_correlated = np.zeros((n_alpha, n_p_values, nsamples_per_p, L, L), dtype=np.float32)
    labels_correlated = np.zeros((n_alpha, n_p_values, nsamples_per_p), dtype=np.uint8)

    vacancies_all = [(x, y) for x in range(L) for y in range(L)]
    nbrs_all = {
        (x, y): find_neighbours(x, y, L) 
        for x in range(L) for y in range(L)
    }

    def correlated_map(p, alpha):
        m = np.zeros((L, L), dtype=np.int8)
        vacancies = vacancies_all.copy()
        occupied_cells = []
        cells_to_fill = int(p * L * L)

        exp_alpha = np.exp(alpha)

        for i in range(cells_to_fill):
            if not vacancies:
                break
                
            probability = np.random.rand()
            
            if probability < exp_alpha / (exp_alpha + i):
                ind = np.random.randint(len(vacancies))
                x, y = vacancies.pop(ind)
            else:
                while occupied_cells:
                    ind = np.random.randint(len(occupied_cells))
                    x0, y0 = occupied_cells[ind]
                    nbrs = [coord for coord in nbrs_all[(x0, y0)] if m[coord] == 0]

                    if not nbrs:
                        occupied_cells.pop(ind)
                    else:
                        x, y = choice(nbrs)
                        vacancies.remove((x, y))
                        break
                else:
                    ind = np.random.randint(len(vacancies))
                    x, y = vacancies.pop(ind)

            m[x, y] = 1
            occupied_cells.append((x, y))

        return m

    # Основной цикл с tqdm для альфа-значений
    for a_idx, alpha in enumerate(tqdm(alpha_values, desc="Alpha значения")):
        for p_idx, p in enumerate(pp):
            for s_idx in range(nsamples_per_p):
                m = correlated_map(p, alpha)
                label_val = dfs(m, L)

                mm_correlated[a_idx, p_idx, s_idx, :, :] = m
                labels_correlated[a_idx, p_idx, s_idx] = label_val

    np.save("maps_correlated.npy", mm_correlated)
    np.save("labels_correlated.npy", labels_correlated)

    print("Сохранено: maps_correlated.npy")
    print("Сохранено: labels_correlated.npy")

    return mm_correlated, labels_correlated

In [12]:
def generate_data(generate_random=True, generate_correlated=True):
    print("Генерация карт:")
    results = {}

    if generate_random:
        mm_random, labels_random = random_maps()
        results['random'] = (mm_random, labels_random)

    if generate_correlated:
        mm_corr, labels_corr = correlated_maps()
        results['correlated'] = (mm_corr, labels_corr)

    print("\nВсе карты сгенерировались")
    return results

In [13]:
generate_data()

Генерация карт:
Генерация некоррелированных карт:


p значения:   0%|          | 0/40 [00:00<?, ?it/s]

Сохранено: maps_random.npy
Сохранено: labels_random.npy
Генерация коррелированных карт:


Alpha значения:   0%|          | 0/4 [00:00<?, ?it/s]

Сохранено: maps_correlated.npy
Сохранено: labels_correlated.npy

Все карты сгенерировались


{'random': (array([[[[0., 0., 0., ..., 0., 1., 0.],
           [0., 1., 0., ..., 0., 0., 0.],
           [0., 1., 0., ..., 0., 1., 1.],
           ...,
           [0., 1., 1., ..., 0., 0., 0.],
           [0., 0., 1., ..., 0., 0., 1.],
           [1., 0., 0., ..., 1., 1., 0.]],
  
          [[0., 0., 0., ..., 0., 0., 0.],
           [0., 1., 0., ..., 1., 1., 0.],
           [1., 1., 0., ..., 1., 1., 0.],
           ...,
           [1., 1., 0., ..., 0., 0., 0.],
           [1., 1., 1., ..., 1., 1., 0.],
           [0., 0., 1., ..., 0., 1., 1.]],
  
          [[0., 0., 1., ..., 1., 1., 1.],
           [0., 1., 0., ..., 1., 1., 0.],
           [0., 0., 0., ..., 1., 0., 1.],
           ...,
           [0., 0., 1., ..., 0., 1., 0.],
           [0., 0., 0., ..., 0., 0., 0.],
           [1., 1., 0., ..., 0., 1., 0.]],
  
          ...,
  
          [[1., 0., 1., ..., 0., 0., 0.],
           [0., 0., 1., ..., 0., 1., 0.],
           [0., 0., 1., ..., 1., 1., 1.],
           ...,
           [0.

In [14]:
def combining_files():
    print("Загрузка данных")
    mm_random = np.load("maps_random.npy")
    labels_random = np.load("labels_random.npy")
    mm_correlated = np.load("maps_correlated.npy")
    labels_correlated = np.load("labels_correlated.npy")

    random_flat = mm_random.reshape(-1, L, L)
    random_labels_flat = labels_random.reshape(-1)
    correlated_flat = mm_correlated.reshape(-1, L, L)
    correlated_labels_flat = labels_correlated.reshape(-1)

    print("Объединение")
    all_maps = np.concatenate([random_flat, correlated_flat], axis=0)
    all_labels = np.concatenate([random_labels_flat, correlated_labels_flat], axis=0)

    print("Перемешивание")
    indices = np.random.permutation(len(all_maps))
    all_maps = all_maps[indices]
    all_labels = all_labels[indices]

    print("Сохранение")
    np.save("maps.npy", all_maps)
    np.save("labels.npy", all_labels)

    return all_maps, all_labels

In [15]:
all_maps, all_labels = combining_files()

Загрузка данных
Объединение
Перемешивание
Сохранение


In [16]:
def signs(map_matrix):
    signs_map = []

    signs_map.append(np.mean(map_matrix))
    signs_map.append(np.std(map_matrix))
    signs_map.append(np.sum(map_matrix))

    row_means = np.mean(map_matrix, axis=1)
    col_means = np.mean(map_matrix, axis=0)
    signs_map.extend([
        np.mean(row_means),
        np.std(row_means),
        np.mean(col_means),
        np.std(col_means)
    ])

    try:
        horiz_corr = np.corrcoef(map_matrix[:, :-1].flatten(), map_matrix[:, 1:].flatten())[0, 1]
    except:
        horiz_corr = 0

    try:
        vert_corr = np.corrcoef(map_matrix[:-1, :].flatten(), map_matrix[1:, :].flatten())[0, 1]
    except:
        vert_corr = 0
    signs_map.extend([horiz_corr, vert_corr])

    labeled_array, num_features = label(map_matrix > 0.5)
    signs_map.append(num_features)

    if num_features > 0:
        cluster_sizes = [np.sum(labeled_array == i) for i in range(1, num_features + 1)]
        signs_map.append(max(cluster_sizes))
        signs_map.append(np.mean(cluster_sizes))
    else:
        signs_map.extend([0, 0])

    left_col_open = np.sum(map_matrix[:, 0] > 0.5)
    right_col_open = np.sum(map_matrix[:, -1] > 0.5)
    signs_map.extend([left_col_open, right_col_open])

    h, w = map_matrix.shape
    left_part = map_matrix[:, :w//3]
    center_part = map_matrix[:, w//3:2*w//3]
    right_part = map_matrix[:, 2*w//3:]

    signs_map.append(np.mean(left_part))
    signs_map.append(np.mean(center_part))
    signs_map.append(np.mean(right_part))

    middle_rows = map_matrix[h//4:3*h//4, :]
    signs_map.append(np.mean(middle_rows))

    for slice_idx in [w//4, w//2, 3*w//4]:
        slice_col = map_matrix[:, slice_idx]
        signs_map.append(np.sum(slice_col > 0.5))

    return np.array(signs_map)

def signs_all_maps(maps_array):
    print(f"Извлечение признаков из {len(maps_array):,} карт")
    X_signs = []

    for i in tqdm(range(len(maps_array)), desc="Обработка карт"):
        features = signs(maps_array[i])
        X_signs.append(features)

    return np.array(X_signs)

In [17]:
def train(X_train, y_train):
    rf_model = RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_split=8,
        min_samples_leaf=4,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
        verbose=0
    )

    print(f"Обучение на {len(X_train):,} образцах")
    rf_model.fit(X_train, y_train)

    joblib.dump(rf_model, MODEL_PATH)
    print(f"\nМодель сохранена в '{MODEL_PATH}'")

    return rf_model

In [18]:
def train_model():
    print("1. Загрузка данных")
    all_maps = np.load("maps.npy")
    all_labels = np.load("labels.npy")

    print(f"Загружено: {all_maps.shape[0]:,} карт")

    print("\n2. Извлечение признаков")
    X_signs = signs_all_maps(all_maps)
    print(f"Получено {X_signs.shape[1]} признаков")

    print("\n3. Разделение данных")
    X_train, X_test, y_train, y_test = train_test_split(
        X_signs, all_labels,
        test_size=0.2,
        random_state=42,
        stratify=all_labels
    )

    print("\n4. Обучение")
    rf_model = train(X_train, y_train)

    print("\n5. Точность")
    y_test_pred = rf_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    print(test_accuracy)
    return rf_model

In [19]:
model = train_model()

1. Загрузка данных
Загружено: 200,000 карт

2. Извлечение признаков
Извлечение признаков из 200,000 карт


Обработка карт:   0%|          | 0/200000 [00:00<?, ?it/s]

Получено 21 признаков

3. Разделение данных

4. Обучение
Обучение на 160,000 образцах

Модель сохранена в 'random_forest_percolation_model.joblib'

5. Точность
0.93425


In [22]:
def predict(data='Xtest.pickle'):
    print(f"Загрузка модели из {MODEL_PATH}")
    rf_model = joblib.load(MODEL_PATH)

    print(f"\nЗагрузка ваших карт из {data}...")
    with open(data, 'rb') as f:
        X_test = pickle.load(f)

    print("\nИзвлечение признаков из карт")
    X_signs = []
    for i in range(len(X_test)):
        signs_map = signs(X_test[i])
        X_signs.append(signs_map)

    X_signs = np.array(X_signs)

    print("\nПредсказываем")
    predictions = rf_model.predict(X_signs)

    print(f"Получено {len(predictions)} предсказаний")

    results_df = pd.DataFrame({
        'id': range(len(predictions)),
        'label': predictions,
    })

    output_file = 'final.csv'
    results_df.to_csv(output_file, index=False)
    print(f"\nРезультаты сохранены в '{output_file}'")

In [23]:
predict()

Загрузка модели из random_forest_percolation_model.joblib

Загрузка ваших карт из Xtest.pickle...

Извлечение признаков из карт

Предсказываем
Получено 25600 предсказаний

Результаты сохранены в 'final.csv'
